# 🎯 DipRadar — Bootstrap ML (Colab)

Corre cada célula **uma de cada vez**, por ordem.  
O Parquet acumula no Google Drive — podes fechar o Colab entre sessões sem perder progresso.

| Batch | Tickers | Tempo estimado |
|-------|---------|----------------|
| 0–200 | 200 | ~25 min |
| 200–400 | 200 | ~25 min |
| 400–600 | 200 | ~25 min |
| 600–800 | 200 | ~25 min |
| 800–fim | ~80 | ~10 min |
| Treino final | — | ~3 min |

> ⚠️ **Cada sessão nova do Colab**: volta a correr as células 1, 2 e 3 antes do batch.


## 1 · Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## 2 · Instalar dependências

In [ ]:
%pip install -q yfinance scikit-learn pyarrow fastparquet
print('✅ Dependências instaladas')

## 3 · Clonar / actualizar repositório

In [ ]:
import os

REPO_DIR = '/content/DipRadar'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull --quiet
    print('✅ Repositório actualizado')
else:
    !git clone --quiet https://github.com/romeurf/DipRadar.git $REPO_DIR
    %cd $REPO_DIR
    print('✅ Repositório clonado')

DRIVE_DIR = '/content/drive/MyDrive/DipRadar'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'📁 Drive dir: {DRIVE_DIR}')

## 4 · Batch 1 — tickers 0–200

> Corre só uma vez. Se a sessão expirar a meio, volta a correr — os tickers já processados são saltados automaticamente.

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 5 · Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 6 · Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 7 · Batch 4 — tickers 600–800

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 600 800 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 8 · Batch 5 — tickers 800 até ao fim

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 800 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 9 · Treino final

> Só depois de todos os batches estarem completos. Treina o modelo com o Parquet acumulado e guarda o `.pkl` no Drive.

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --skip-backfill \
    --drive-dir /content/drive/MyDrive/DipRadar

## 10 · Verificar ficheiros gerados

In [ ]:
import os, pathlib

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')
print('📁 Ficheiros no Drive:')
for f in sorted(drive_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:>8.1f} KB')

# Mostrar estatísticas do Parquet
import pandas as pd
pq = drive_dir / 'ml_training_price.parquet'
if pq.exists():
    df = pd.read_parquet(pq)
    print(f'\n📊 Parquet: {len(df)} alertas | {df["symbol"].nunique()} tickers únicos')
    print(df['outcome_label'].value_counts().to_string())
else:
    print('⚠️  Parquet ainda não existe')

## 11 · Copiar .pkl para o Railway (opcional)

> Se quiseres usar o modelo no bot imediatamente, copia o `.pkl` do Drive para o Railway via Railway CLI:
> ```
> railway run -- cp /data/dip_model_price.pkl /tmp/  # não funciona directamente
> ```
> A forma correcta é fazer commit do `.pkl` para o repositório (se < 100 MB) ou usar Railway Volumes.
> O bot já lê automaticamente o `.pkl` do `/data` no arranque.